<a href="https://colab.research.google.com/github/rjf7q/neural-network-challenge-2/blob/main/attrition.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Part 1: Preprocessing

In [63]:
# Import our dependencies
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import pandas as pd
import numpy as np
from tensorflow.keras.models import Model
from tensorflow.keras import layers

#  Import and read the attrition data
attrition_df = pd.read_csv('https://static.bc-edx.com/ai/ail-v-1-0/m19/lms/datasets/attrition.csv')
attrition_df.head()

,Age,Attrition,BusinessTravel,Department,DistanceFromHome,Education,EducationField,EnvironmentSatisfaction,HourlyRate,JobInvolvement,...,PerformanceRating,RelationshipSatisfaction,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41,Yes,Travel_Rarely,Sales,1,2,Life Sciences,2,94,3,...,3,1,0,8,0,1,6,4,0,5
1,49,No,Travel_Frequently,Research & Development,8,1,Life Sciences,3,61,2,...,4,4,1,10,3,3,10,7,1,7
2,37,Yes,Travel_Rarely,Research & Development,2,2,Other,4,92,2,...,3,2,0,7,3,3,0,0,0,0
3,33,No,Travel_Frequently,Research & Development,3,4,Life Sciences,4,56,3,...,3,3,0,8,3,3,8,7,3,0
4,27,No,Travel_Rarely,Research & Development,2,1,Medical,1,40,3,...,3,4,1,6,3,3,2,2,2,2


In [64]:
# Determine the number of unique values in each column
attrition_df.nunique()

,0
Age,43
Attrition,2
BusinessTravel,3
Department,3
DistanceFromHome,29
Education,5
EducationField,6
EnvironmentSatisfaction,4
HourlyRate,71
JobInvolvement,4


In [65]:
# Create y_df with the Attrition and Department columns
y_df = attrition_df[['Attrition', 'Department']]

In [66]:
# Create a list of at least 10 column names to use as X data
X = attrition_df.drop(columns=['Attrition', 'Department'])

# Create X_df using your selected columns
X_df=X

# Show the data types for X_df
X_df.dtypes

,0
Age,int64
BusinessTravel,object
DistanceFromHome,int64
Education,int64
EducationField,object
EnvironmentSatisfaction,int64
HourlyRate,int64
JobInvolvement,int64
JobLevel,int64
JobRole,object


In [67]:
# Split the data into training and testing sets
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X_df, y_df, random_state=0)


In [68]:
# Convert your X data to numeric data types however you see fit
# Add new code cells as necessary
X_train = pd.get_dummies(X_train)
X_test = pd.get_dummies(X_test)

In [69]:
# Create a StandardScaler
scaler = StandardScaler()

# Fit the StandardScaler to the training data
X_train_scaled = scaler.fit(X_train)

# Scale the training and testing data
X_train_scaled = X_train_scaled.transform(X_train)
#X_test_scaled = X_train_scaled.transform(X_test)

In [70]:
from sklearn.preprocessing import OneHotEncoder

# Create a OneHotEncoder for the Department column
enc=OneHotEncoder(handle_unknown='ignore')

# Fit the encoder to the training data
enc.fit(y_train['Department'].values.reshape(-1,1))

# Create two new variables by applying the encoder
# to the training and testing data
y_train_enc = enc.transform(y_train['Department'].values.reshape(-1,1))
y_test_enc = enc.transform(y_test['Department'].values.reshape(-1,1))


In [71]:
# Create a OneHotEncoder for the Attrition column
enc=OneHotEncoder(handle_unknown='ignore')

# Fit the encoder to the training data
enc.fit(y_train['Attrition'].values.reshape(-1,1))

# Create two new variables by applying the encoder
# to the training and testing data
y_train_enc = enc.transform(y_train['Attrition'].values.reshape(-1,1))
y_test_enc = enc.transform(y_test['Attrition'].values.reshape(-1,1))

## Part 2: Create, Compile, and Train the Model

In [72]:
# Find the number of columns in the X training data.
X_train.shape[1]

# Create the input layer
input_layer = layers.Input(shape=(X_train.shape[1],))

# Create at least two shared layers
shared_layer1 = layers.Dense(10, activation='relu')(input_layer)
shared_layer2 = layers.Dense(5, activation='relu')(shared_layer1)

In [73]:
# Create a branch for Department
# with a hidden layer and an output layer
department_branch = layers.Dense(10, activation='relu')(shared_layer2)
department_output = layers.Dense(1, activation='sigmoid')(department_branch)
# Create the hidden layer
department_hidden = layers.Dense(10, activation='relu')(shared_layer2)

# Create the output layer
department_output = layers.Dense(1, activation='sigmoid')(department_hidden)

In [74]:
# Create a branch for Attrition
# with a hidden layer and an output layer
attrition_branch = layers.Dense(10, activation='relu')(shared_layer2)
attrition_output = layers.Dense(1, activation='sigmoid')(attrition_branch)
# Create the hidden layer
attrition_hidden = layers.Dense(10, activation='relu')(shared_layer2)

# Create the output layer
attrition_output = layers.Dense(1, activation='sigmoid')(attrition_hidden)

In [75]:
# Create the model
model = Model(inputs=input_layer, outputs=[department_output, attrition_output])

# Compile the model
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Summarize the model
model.summary()

Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)              ┃ Output Shape           ┃        Param # ┃ Connected to           ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer_3             │ (None, 43)             │              0 │ -                      │
│ (InputLayer)              │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dense_30 (Dense)          │ (None, 10)             │            440 │ input_layer_3[0][0]    │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dense_31 (Dense)          │ (None, 5)              │             55 │ dense_30[0][0]         │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dense_34 (Dense)          │ (None, 10)             │             60 │ dense_31[0][0]         │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dense_38 (Dense)          │ (None, 10)             │             60 │ dense_31[0][0]         │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dense_35 (Dense)          │ (None, 1)              │             11 │ dense_34[0][0]         │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dense_39 (Dense)          │ (None, 1)              │             11 │ dense_38[0][0]         │
└───────────────────────────┴────────────────────────┴────────────────┴────────────────────────┘

 Total params: 637 (2.49 KB)

 Trainable params: 637 (2.49 KB)

 Non-trainable params: 0 (0.00 B)

In [76]:
# Train the model
#model.fit(X_train_scaled, [y_train_enc, y_train_enc], epochs=100, batch_size=32)

Epoch 1/100


TypeError: Failed to convert elements of SparseTensor(indices=Tensor("data_1:0", shape=(None, 2), dtype=int64), values=Tensor("data_2:0", shape=(None,), dtype=float32), dense_shape=Tensor("data_3:0", shape=(2,), dtype=int64)) to Tensor. Consider casting elements to a supported type. See https://www.tensorflow.org/api_docs/python/tf/dtypes for supported TF dtypes.

In [ ]:
# Evaluate the model with the testing data
#test_loss, test_accuracy = model.evaluate(X_test_scaled, [y_test_enc, y_test_enc])
#print(f"Test Loss: {test_loss}, Test Accuracy: {test_accuracy}")

In [ ]:
# Print the accuracy for both department and attrition
#print(f"Department Accuracy: {test_accuracy[0]}")
#print(f"Attrition Accuracy: {test_accuracy[1]}")

# Summary

In the provided space below, briefly answer the following questions.

1. Is accuracy the best metric to use on this data? Why or why not?

2. What activation functions did you choose for your output layers, and why?

3. Can you name a few ways that this model might be improved?

YOUR ANSWERS HERE

1.
2.
3.